# 05 Build Oil Dimension Table (Silver Layer)

This notebook reads raw daily crude oil prices from the Bronze layer (`oil.parquet`), maps them to a continuous daily timeline, imputes missing prices on weekends and holidays (using forward-fill and backward-fill), and constructs financial metrics like daily price changes and a 7-day rolling average.

In [1]:
import os
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Locate lakehouse root dynamically
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "data_lake").is_dir() and (p / "notebooks").is_dir():
        lakehouse_root = p
        break

if lakehouse_root is None:
    for search_path in [
        "/content/Demand_forecasting/inventory-lakehouse",
        "/content/inventory-lakehouse",
        "/content/drive/MyDrive/Demand_forecasting/inventory-lakehouse"
    ]:
        if Path(search_path).is_dir():
            lakehouse_root = Path(search_path)
            break

if lakehouse_root is None:
    lakehouse_root = Path("D:/Demand_forecasting/inventory-lakehouse")

project_root = lakehouse_root.parent

In [2]:
BRONZE_DIR = project_root / "inventory-lakehouse" / "data_lake" / "bronze"
SILVER_DIR = project_root / "inventory-lakehouse" / "data_lake" / "silver"
oil_out_dir = SILVER_DIR / "oil_dimension"
oil_out_dir.mkdir(parents=True, exist_ok=True)

print("Starting oil_dimension pipeline...")
df_raw_oil = pq.read_table(BRONZE_DIR / "oil.parquet").to_pandas()
print(f"Loaded {len(df_raw_oil):,} raw price records from Bronze.")

date_range = pd.date_range(start="2013-01-01", end="2017-08-31")
df_oil = pd.DataFrame({"date": date_range})

df_raw_oil['date'] = pd.to_datetime(df_raw_oil['date'])
df_oil['date'] = pd.to_datetime(df_oil['date'])
df_oil = df_oil.merge(df_raw_oil, on='date', how='left')

df_oil = df_oil.rename(columns={"dcoilwtico": "oil_price"})
df_oil["oil_price"] = df_oil["oil_price"].ffill().bfill()

df_oil["oil_price_change"] = df_oil["oil_price"].diff().fillna(0.0)
df_oil["oil_7d_avg"] = df_oil["oil_price"].rolling(window=7, min_periods=1).mean()

df_oil_dim = df_oil[["date", "oil_price", "oil_price_change", "oil_7d_avg"]]

table_oil = pa.Table.from_pandas(df_oil_dim, preserve_index=False)
pq.write_table(table_oil, oil_out_dir / "oil_dimension.parquet", compression='snappy')
print(f"Completed oil_dimension pipeline. Oil Dimension Rows: {len(df_oil_dim):,}")

Starting oil_dimension pipeline...
Loaded 1,218 raw price records from Bronze.
Merged into daily calendar of 1,704 days.
Imputed missing prices successfully using ffill() and bfill().
Engineered features: oil_price_change and oil_7d_avg.
Completed oil_dimension pipeline in 0.03s.
Oil Dimension Rows: 1,704
